# Chapter 3.3: Multi-Interest Retrieval

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** why single-vector user representations are insufficient for diverse interests
2. **Implement** MIND's multi-interest extraction with dynamic routing
3. **Describe** ComiRec's controllable multi-interest framework
4. **Build** capsule network-based multi-head interest extraction
5. **Understand** SDM (Sequential Deep Matching) for long and short-term interests
6. **Compare** single-interest vs multi-interest retrieval quality
7. **Apply** dynamic routing mechanism for clustering user behaviors into interest groups

## Prerequisites

- Two-tower retrieval models (Chapter 3.1)
- Basic understanding of attention mechanisms
- PyTorch fundamentals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part3/chapter_3.3_multi_interest.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part3/chapter_3.3_multi_interest.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")

## 1. The Multi-Interest Problem

A single user embedding $\mathbf{u} \in \mathbb{R}^d$ must represent ALL of a user's diverse interests. Consider a user who likes:
- Sci-fi movies
- Cooking recipes
- Rock climbing gear

A single vector is the weighted average of these interests — it may not be close to ANY individual interest cluster in embedding space.

**Multi-interest models** represent each user with $K$ embedding vectors:

$$\mathbf{U} = \{\mathbf{u}_1, \mathbf{u}_2, \ldots, \mathbf{u}_K\}$$

Retrieval score becomes:

$$s(u, i) = \max_{k \in [K]} \mathbf{u}_k^\top \mathbf{v}_i$$

> **💡 Concept:** The "interest collapse" problem: when you average diverse interests into one vector, you get a point equidistant from all clusters but near none. Multi-interest models solve this by maintaining separate interest vectors.

In [ ]:
# Visualize the interest collapse problem
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Generate 3 interest clusters
np.random.seed(42)
cluster_centers = np.array([[2, 3], [-3, 2], [1, -3]])
cluster_labels = ['Sci-Fi', 'Cooking', 'Sports']
cluster_colors = ['#e74c3c', '#2ecc71', '#3498db']

points = []
for center in cluster_centers:
    pts = center + np.random.randn(20, 2) * 0.5
    points.append(pts)

# Single interest: average of all
all_points = np.concatenate(points)
single_interest = all_points.mean(axis=0)

for ax_idx, ax in enumerate(axes):
    for i, (pts, label, color) in enumerate(zip(points, cluster_labels, cluster_colors)):
        ax.scatter(pts[:, 0], pts[:, 1], c=color, s=30, alpha=0.6, label=label)
    
    if ax_idx == 0:
        ax.scatter(*single_interest, marker='*', s=300, c='black', zorder=5, 
                   label='Single Interest Vector')
        for center in cluster_centers:
            dist = np.linalg.norm(single_interest - center)
            ax.annotate(f'd={dist:.1f}', xy=((single_interest[0]+center[0])/2, 
                        (single_interest[1]+center[1])/2), fontsize=9)
            ax.plot([single_interest[0], center[0]], [single_interest[1], center[1]], 
                    'k--', alpha=0.3)
        ax.set_title('Single Interest (Collapsed)', fontsize=13)
    else:
        multi_interests = np.array([pts.mean(axis=0) for pts in points])
        for mi, color in zip(multi_interests, cluster_colors):
            ax.scatter(*mi, marker='*', s=300, c=color, edgecolors='black', 
                       linewidths=1.5, zorder=5)
        ax.set_title('Multi-Interest (3 Vectors)', fontsize=13)
    
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')

plt.suptitle('Single vs. Multi-Interest User Representation', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. MIND: Multi-Interest Network with Dynamic Routing (Alibaba, 2019)

MIND (Li et al., 2019, Alibaba) uses **capsule networks with dynamic routing** to extract multiple interest vectors from a user's behavior sequence.

### Dynamic Routing Algorithm

Given behavior embeddings $\{\mathbf{e}_1, \ldots, \mathbf{e}_n\}$ and $K$ interest capsules:

1. Initialize routing logits $b_{ij} = 0$
2. For $r = 1, \ldots, R$ routing iterations:
   - Compute coupling coefficients: $c_{ij} = \text{softmax}_j(b_{ij})$
   - Compute capsule input: $\mathbf{s}_j = \sum_i c_{ij} \mathbf{W}_j \mathbf{e}_i$
   - Apply squash: $\mathbf{v}_j = \text{squash}(\mathbf{s}_j) = \frac{\|\mathbf{s}_j\|^2}{1 + \|\mathbf{s}_j\|^2} \cdot \frac{\mathbf{s}_j}{\|\mathbf{s}_j\|}$
   - Update routing: $b_{ij} \leftarrow b_{ij} + \mathbf{e}_i^\top \mathbf{W}_j^\top \mathbf{v}_j$

The $K$ output capsules $\{\mathbf{v}_1, \ldots, \mathbf{v}_K\}$ become the user's multi-interest vectors.

> **⚠️ Common Pitfall:** The number of interests $K$ is a hyperparameter that affects both quality and serving cost. Too few interests miss diversity; too many increase ANN lookup cost ($K$ queries per user). Typical values: $K \in [4, 8]$.

In [ ]:
def squash(tensor: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Squash activation for capsule networks.
    
    Maps vectors to length in (0, 1) while preserving direction.
    """
    squared_norm = (tensor ** 2).sum(dim=dim, keepdim=True)
    scale = squared_norm / (1 + squared_norm)
    return scale * tensor / (torch.sqrt(squared_norm) + 1e-8)


class DynamicRoutingLayer(nn.Module):
    """Dynamic routing for multi-interest extraction (MIND)."""
    
    def __init__(self, input_dim: int, output_dim: int, num_interests: int = 4,
                 num_iterations: int = 3):
        super().__init__()
        self.num_interests = num_interests
        self.num_iterations = num_iterations
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # Transformation matrix for each capsule
        self.W = nn.Parameter(torch.randn(num_interests, input_dim, output_dim) * 0.01)
    
    def forward(self, behavior_emb: torch.Tensor, 
                mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            behavior_emb: (B, L, D_in) behavior sequence embeddings
            mask: (B, L) boolean mask for valid behaviors
        
        Returns:
            interests: (B, K, D_out) multi-interest vectors
        """
        B, L, D_in = behavior_emb.shape
        K = self.num_interests
        
        # Transform behavior embeddings for each capsule
        # (B, L, D_in) @ (K, D_in, D_out) -> (B, K, L, D_out)
        u_hat = torch.einsum('bld,kdo->bklo', behavior_emb, self.W)
        
        # Initialize routing logits
        b = torch.zeros(B, K, L, device=behavior_emb.device)
        
        # Apply mask to routing logits
        if mask is not None:
            mask_expanded = mask.unsqueeze(1).expand(-1, K, -1)  # (B, K, L)
            b = b.masked_fill(~mask_expanded, -1e9)
        
        # Dynamic routing iterations
        for iteration in range(self.num_iterations):
            # Coupling coefficients
            c = F.softmax(b, dim=1)  # (B, K, L) - softmax over capsules
            
            # Weighted sum of transformed behaviors
            s = torch.einsum('bkl,bkld->bkd', c, u_hat)  # (B, K, D_out)
            
            # Squash
            v = squash(s, dim=-1)  # (B, K, D_out)
            
            # Update routing logits (except last iteration)
            if iteration < self.num_iterations - 1:
                delta = torch.einsum('bkld,bkd->bkl', u_hat, v)  # (B, K, L)
                b = b + delta
        
        return v  # (B, K, D_out)


# Test
B, L, D_in, D_out, K = 4, 20, 32, 32, 4
routing = DynamicRoutingLayer(D_in, D_out, num_interests=K, num_iterations=3)

behavior = torch.randn(B, L, D_in)
mask = torch.ones(B, L, dtype=torch.bool)
mask[:, 15:] = False  # last 5 positions are padding

interests = routing(behavior, mask)
print(f"Input behavior shape:  {behavior.shape}")
print(f"Output interests shape: {interests.shape}")
print(f"Interest norms: {interests.norm(dim=-1)[0].tolist()}")

In [ ]:
class MIND(nn.Module):
    """Multi-Interest Network with Dynamic Routing (Li et al., 2019)."""
    
    def __init__(self, num_items: int, embed_dim: int = 32, 
                 num_interests: int = 4, num_routing_iter: int = 3):
        super().__init__()
        self.num_interests = num_interests
        self.embed_dim = embed_dim
        
        # Item embedding (shared between user history and target item)
        self.item_embedding = nn.Embedding(num_items, embed_dim, padding_idx=0)
        
        # Dynamic routing for multi-interest extraction
        self.routing = DynamicRoutingLayer(
            input_dim=embed_dim, output_dim=embed_dim,
            num_interests=num_interests, num_iterations=num_routing_iter
        )
    
    def get_user_interests(self, history: torch.Tensor, 
                           mask: torch.Tensor) -> torch.Tensor:
        """Extract multi-interest vectors from user history.
        
        Args:
            history: (B, L) item ID sequence
            mask: (B, L) boolean mask
        
        Returns:
            (B, K, D) multi-interest embeddings
        """
        behavior_emb = self.item_embedding(history)  # (B, L, D)
        interests = self.routing(behavior_emb, mask)  # (B, K, D)
        return F.normalize(interests, p=2, dim=-1)
    
    def get_item_embedding(self, item_ids: torch.Tensor) -> torch.Tensor:
        """Get item embeddings."""
        return F.normalize(self.item_embedding(item_ids), p=2, dim=-1)
    
    def forward(self, history: torch.Tensor, mask: torch.Tensor,
                target_items: torch.Tensor) -> torch.Tensor:
        """Compute scores for target items.
        
        Score = max over interests of cosine similarity.
        """
        interests = self.get_user_interests(history, mask)  # (B, K, D)
        target_emb = self.get_item_embedding(target_items)  # (B, D)
        
        # Score = max over interests
        # (B, K, D) @ (B, D, 1) -> (B, K, 1) -> (B, K)
        scores = torch.bmm(interests, target_emb.unsqueeze(-1)).squeeze(-1)
        max_score = scores.max(dim=-1).values  # (B,)
        return max_score, scores


# Create synthetic data with clear interest clusters
NUM_ITEMS = 5000
NUM_CLUSTERS = 5
items_per_cluster = NUM_ITEMS // NUM_CLUSTERS

model = MIND(NUM_ITEMS + 1, embed_dim=32, num_interests=4)
print(f"MIND parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. ComiRec: Controllable Multi-Interest Retrieval

ComiRec (Cen et al., 2020) extends MIND with two key innovations:

### ComiRec-DR (Dynamic Routing)
Same dynamic routing as MIND but with improved training objectives.

### ComiRec-SA (Self-Attention)
Uses multi-head self-attention instead of dynamic routing:

$$\mathbf{V}_k = \text{softmax}\left(\frac{\mathbf{w}_k^\top \tanh(\mathbf{W}_1 \mathbf{H})}{\sqrt{d}}\right) \mathbf{H}$$

where $\mathbf{H}$ is the behavior matrix and $\mathbf{w}_k$ is a learnable query for interest $k$.

### Controllable Diversity
ComiRec uses a diversity-aware aggregation at inference:

$$\text{Score}(i) = \max_k \mathbf{v}_k^\top \mathbf{e}_i - \lambda \max_{j \in S} \text{sim}(\mathbf{e}_i, \mathbf{e}_j)$$

where $S$ is the set of already selected items and $\lambda$ controls diversity.

> **🔑 Pro Tip:** ComiRec-SA is generally faster and simpler than ComiRec-DR while achieving similar performance. The self-attention version is easier to optimize and parallelize.

In [ ]:
class ComiRecSA(nn.Module):
    """ComiRec with Self-Attention (Cen et al., 2020)."""
    
    def __init__(self, num_items: int, embed_dim: int = 32, num_interests: int = 4):
        super().__init__()
        self.num_interests = num_interests
        self.embed_dim = embed_dim
        
        self.item_embedding = nn.Embedding(num_items, embed_dim, padding_idx=0)
        
        # Multi-head attention for interest extraction
        # Each head learns to attend to a different interest
        self.W1 = nn.Linear(embed_dim, embed_dim, bias=False)
        self.interest_queries = nn.Parameter(torch.randn(num_interests, embed_dim) * 0.1)
    
    def extract_interests(self, history: torch.Tensor, 
                          mask: torch.Tensor) -> torch.Tensor:
        """Extract multi-interest vectors using self-attention."""
        B, L = history.shape
        behavior_emb = self.item_embedding(history)  # (B, L, D)
        
        # Project behaviors
        H = torch.tanh(self.W1(behavior_emb))  # (B, L, D)
        
        # Attention weights for each interest
        # (K, D) @ (B, D, L) -> (B, K, L)
        attn_logits = torch.einsum('kd,bld->bkl', self.interest_queries, H)
        attn_logits = attn_logits / (self.embed_dim ** 0.5)
        
        # Mask padding
        if mask is not None:
            mask_expanded = mask.unsqueeze(1).expand(-1, self.num_interests, -1)
            attn_logits = attn_logits.masked_fill(~mask_expanded, -1e9)
        
        attn_weights = F.softmax(attn_logits, dim=-1)  # (B, K, L)
        
        # Weighted sum
        interests = torch.bmm(attn_weights, behavior_emb)  # (B, K, D)
        return F.normalize(interests, p=2, dim=-1), attn_weights
    
    def forward(self, history, mask, target_items):
        interests, attn_weights = self.extract_interests(history, mask)
        target_emb = F.normalize(self.item_embedding(target_items), p=2, dim=-1)
        
        scores = torch.bmm(interests, target_emb.unsqueeze(-1)).squeeze(-1)
        max_score = scores.max(dim=-1).values
        return max_score, scores, attn_weights


# Compare MIND vs ComiRec-SA
comirec = ComiRecSA(NUM_ITEMS + 1, embed_dim=32, num_interests=4)

# Test with synthetic behavior
B, L = 8, 30
history = torch.randint(1, NUM_ITEMS, (B, L))
mask = torch.ones(B, L, dtype=torch.bool)
mask[:, 25:] = False
target = torch.randint(1, NUM_ITEMS, (B,))

score_mind, all_scores_mind = model(history, mask, target)
score_comi, all_scores_comi, attn = comirec(history, mask, target)

print(f"MIND - Max score: {score_mind.mean():.4f}, All scores: {all_scores_mind.shape}")
print(f"ComiRec-SA - Max score: {score_comi.mean():.4f}, All scores: {all_scores_comi.shape}")
print(f"Attention weights shape: {attn.shape}")

In [ ]:
# Visualize attention weights across interests
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sample_attn = attn[0].detach().numpy()  # (K, L)
for k in range(4):
    ax = axes[k // 2, k % 2]
    ax.bar(range(L), sample_attn[k, :L], color=f'C{k}', alpha=0.7)
    ax.set_title(f'Interest {k+1} Attention', fontsize=12)
    ax.set_xlabel('Position in History')
    ax.set_ylabel('Attention Weight')
    ax.set_xlim(-0.5, L - 0.5)

plt.suptitle('ComiRec-SA: Attention Distribution per Interest Head', fontsize=14)
plt.tight_layout()
plt.show()

## 4. SDM: Sequential Deep Matching (Alibaba)

SDM (Lv et al., 2019, Alibaba) captures both **long-term** and **short-term** user preferences:

- **Short-term**: Recent session behavior encoded with LSTM
- **Long-term**: Historical behavior encoded with attention-weighted aggregation
- **Fusion**: Gate mechanism combines both representations

$$\mathbf{u}_{\text{short}} = \text{LSTM}(\mathbf{e}_{t-k}, \ldots, \mathbf{e}_{t-1})$$

$$\mathbf{u}_{\text{long}} = \sum_i \alpha_i \mathbf{e}_i, \quad \alpha_i = \frac{\exp(\mathbf{e}_i^\top \mathbf{u}_{\text{short}})}{\sum_j \exp(\mathbf{e}_j^\top \mathbf{u}_{\text{short}})}$$

$$\mathbf{u} = \text{Gate}(\mathbf{u}_{\text{short}}, \mathbf{u}_{\text{long}})$$

In [ ]:
class SDM(nn.Module):
    """Sequential Deep Matching (Lv et al., 2019)."""
    
    def __init__(self, num_items: int, embed_dim: int = 32, 
                 short_len: int = 10, long_len: int = 50):
        super().__init__()
        self.embed_dim = embed_dim
        self.short_len = short_len
        self.long_len = long_len
        
        self.item_embedding = nn.Embedding(num_items, embed_dim, padding_idx=0)
        
        # Short-term encoder (LSTM)
        self.lstm = nn.LSTM(embed_dim, embed_dim, batch_first=True)
        
        # Long-term encoder (attention)
        self.long_attn = nn.Linear(embed_dim, embed_dim)
        
        # Gate to fuse short and long term
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Sigmoid()
        )
        
        self.output_layer = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, short_history: torch.Tensor, long_history: torch.Tensor,
                short_mask: torch.Tensor, long_mask: torch.Tensor) -> torch.Tensor:
        """Encode user with short-term and long-term preferences."""
        # Short-term: LSTM over recent behaviors
        short_emb = self.item_embedding(short_history)  # (B, S, D)
        lstm_out, (h_n, _) = self.lstm(short_emb)  # h_n: (1, B, D)
        u_short = h_n.squeeze(0)  # (B, D)
        
        # Long-term: attention weighted by short-term
        long_emb = self.item_embedding(long_history)  # (B, L, D)
        attn_query = self.long_attn(u_short)  # (B, D)
        attn_scores = torch.bmm(long_emb, attn_query.unsqueeze(-1)).squeeze(-1)  # (B, L)
        if long_mask is not None:
            attn_scores = attn_scores.masked_fill(~long_mask, -1e9)
        attn_weights = F.softmax(attn_scores, dim=-1)  # (B, L)
        u_long = torch.bmm(attn_weights.unsqueeze(1), long_emb).squeeze(1)  # (B, D)
        
        # Gate fusion
        gate_input = torch.cat([u_short, u_long], dim=-1)  # (B, 2D)
        gate_value = self.gate(gate_input)  # (B, D)
        u_fused = gate_value * u_short + (1 - gate_value) * u_long  # (B, D)
        
        output = self.output_layer(u_fused)
        return F.normalize(output, p=2, dim=-1)


sdm = SDM(NUM_ITEMS + 1, embed_dim=32)
short_hist = torch.randint(1, NUM_ITEMS, (B, 10))
long_hist = torch.randint(1, NUM_ITEMS, (B, 50))
short_mask = torch.ones(B, 10, dtype=torch.bool)
long_mask = torch.ones(B, 50, dtype=torch.bool)
long_mask[:, 40:] = False

user_emb = sdm(short_hist, long_hist, short_mask, long_mask)
print(f"SDM user embedding shape: {user_emb.shape}")
print(f"SDM parameters: {sum(p.numel() for p in sdm.parameters()):,}")

## 5. Training MIND on Synthetic Multi-Interest Data

Let's generate data where users have distinct interest clusters and train MIND to discover them.

In [ ]:
# Generate multi-interest synthetic data
np.random.seed(42)
NUM_USERS = 500
NUM_ITEMS_DATA = 3000
NUM_INTEREST_GROUPS = 5
ITEMS_PER_GROUP = NUM_ITEMS_DATA // NUM_INTEREST_GROUPS

# Each user has 2-3 interest groups
user_interests = {}
user_histories_multi = {}

for u in range(NUM_USERS):
    n_interests = np.random.choice([2, 3])
    interests = np.random.choice(NUM_INTEREST_GROUPS, n_interests, replace=False)
    user_interests[u] = interests
    
    history = []
    for _ in range(np.random.randint(15, 40)):
        # Pick a random interest, then a random item from that interest group
        group = np.random.choice(interests)
        item = group * ITEMS_PER_GROUP + np.random.randint(1, ITEMS_PER_GROUP)
        history.append(item + 1)  # +1 because 0 is padding
    user_histories_multi[u] = history

# Create training pairs (user_history -> next_item)
from torch.utils.data import Dataset, DataLoader

class MultiInterestDataset(Dataset):
    def __init__(self, user_histories, max_len=30):
        self.samples = []
        self.max_len = max_len
        for u, history in user_histories.items():
            for t in range(1, len(history)):
                hist = history[:t][-max_len:]
                target = history[t]
                self.samples.append((hist, target))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        hist, target = self.samples[idx]
        padded = hist + [0] * (self.max_len - len(hist))
        mask = [True] * len(hist) + [False] * (self.max_len - len(hist))
        return {
            'history': torch.tensor(padded, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.bool),
            'target': torch.tensor(target, dtype=torch.long)
        }

train_dataset = MultiInterestDataset(user_histories_multi)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Example user interests: {user_interests[0]}")
print(f"Example history length: {len(user_histories_multi[0])}")

In [ ]:
# Train MIND
mind_model = MIND(NUM_ITEMS_DATA + 2, embed_dim=32, num_interests=4, num_routing_iter=3)
optimizer = torch.optim.Adam(mind_model.parameters(), lr=1e-3)

losses = []
mind_model.train()

for epoch in range(20):
    epoch_loss = 0
    n_batch = 0
    for batch in train_loader:
        interests = mind_model.get_user_interests(batch['history'], batch['mask'])  # (B, K, D)
        target_emb = mind_model.get_item_embedding(batch['target'])  # (B, D)
        
        # Sample random negatives for in-batch style loss
        all_item_emb = target_emb  # Use other targets as negatives (in-batch)
        
        # For each user, best interest should match target
        # Score: max over interests of dot product with target
        scores_per_interest = torch.bmm(interests, target_emb.unsqueeze(-1)).squeeze(-1)  # (B, K)
        best_interest_idx = scores_per_interest.argmax(dim=-1)  # (B,)
        
        # Get best interest embedding for each user
        best_interest = interests[torch.arange(interests.size(0)), best_interest_idx]  # (B, D)
        
        # In-batch contrastive loss using best interest
        logits = torch.matmul(best_interest, target_emb.T) * 20.0  # (B, B)
        labels = torch.arange(logits.size(0))
        loss = F.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batch += 1
    
    avg_loss = epoch_loss / n_batch
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses, 'o-', color='darkorange', linewidth=2, markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('MIND Training Loss', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement MIND with Label-Aware Attention

MIND uses a label-aware attention mechanism during training: the target item is used to select the best interest for computing the loss. Implement this.

In [ ]:
class MINDWithLabelAware(nn.Module):
    """MIND with label-aware attention for training."""
    
    def __init__(self, num_items, embed_dim=32, num_interests=4):
        super().__init__()
        # TODO: Define item embedding, dynamic routing
        # TODO: Define label-aware attention: given target item,
        #       compute attention over interests to select the best one
        pass
    
    def forward(self, history, mask, target):
        # TODO: Extract multi-interest vectors
        # TODO: Use target to compute attention over interests
        # TODO: Return weighted interest embedding and loss
        pass

### 🏋️ Exercise 2: Implement ComiRec Diversity Control

Implement the greedy diversity-aware selection from ComiRec: iteratively select items that are relevant but different from already selected ones.

In [ ]:
def diverse_retrieval(interest_vectors, item_embeddings, top_k=20, 
                      lambda_diversity=0.5):
    """
    Diversity-aware retrieval using MMR-style greedy selection.
    
    Args:
        interest_vectors: (K, D) multi-interest user embeddings
        item_embeddings: (N, D) all item embeddings
        top_k: number of items to retrieve
        lambda_diversity: trade-off between relevance and diversity
    
    Returns:
        selected_ids: list of selected item indices
    """
    # TODO: Compute relevance scores: max over interests
    # TODO: Greedily select items:
    #   1. Pick highest relevance item
    #   2. For each remaining candidate, compute:
    #      score = relevance - lambda * max_similarity_to_selected
    #   3. Pick highest score
    #   4. Repeat until top_k items selected
    pass

### 🏋️ Exercise 3: Compare Single vs Multi-Interest Retrieval

Train both a single-interest two-tower model and MIND on the same data. Compare Recall@K, especially for users with diverse interests.

In [ ]:
def compare_single_vs_multi(train_loader, num_items, embed_dim=32, 
                            num_interests=4, num_epochs=15):
    """
    Compare single-interest and multi-interest models.
    
    Train both models on the same data and evaluate:
    - Overall Recall@10, 50, 100
    - Recall broken down by number of user interests
    - Diversity of retrieved items (category coverage)
    
    Returns:
        Dict with metrics for each model
    """
    # TODO: Train single-interest model (average pooling of history)
    # TODO: Train MIND model
    # TODO: Evaluate both on test set
    # TODO: Plot comparison
    pass

## Summary

| Model | Method | Pros | Cons |
|-------|--------|------|------|
| **MIND** | Dynamic routing capsules | Discovers natural interest clusters | Slow routing iterations |
| **ComiRec-SA** | Self-attention heads | Fast, controllable diversity | Fixed number of interests |
| **ComiRec-DR** | Dynamic routing + diversity | Best of both worlds | More complex training |
| **SDM** | LSTM + attention fusion | Captures temporal dynamics | Single vector output |

**Key insight**: Multi-interest retrieval is especially valuable when users have diverse, non-overlapping interests. The overhead of $K$ ANN queries per user is usually worth the improved recall.

**Next up**: Chapter 3.4 covers Graph-based Retrieval — using graph structure to enrich item and user representations.